
# Applied Machine Learning, Clustering 1
## Segmenting a bank's customer base: k-means from scratch

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we are coming from

Until now every problem in the course had a **correct answer** stored in a column: the amount the customer bought,
whether or not they repaid the loan. The model learned to predict that column and we measured it by comparing its
predictions with the truth in testing. That is **supervised** learning.

Today the answer column does not exist. A bank has 5,660 customers with their age, occupation, marital status,
education and the products they hold, and the question is different: **what types of customers are there?** Nobody
tells us how many types there are or what they are called. That is **unsupervised** learning, and the most widely used
tool for this question is **clustering**: finding groups of customers that are similar to each other and different
from the rest.

Changing the question changes everything else. There is no testing set to say whether the model got it right, so
evaluation is different (and more uncomfortable); the algorithm works with **distances** between customers, so the way
we turn an occupation or a marital status into numbers decides the result; and in the end the model hands back groups
that **someone has to interpret and name**, because the algorithm does not know what a retiree is.

The central idea of the notebook is that conversion. A clustering algorithm does not understand that a married customer
is more like a divorced one than like a single one, or that a manager and a technician have more in common than a
manager and a student. That similarity, obvious to anyone at the bank, has to be **written down by hand**: translating
each category into a score on an axis with business meaning, declaring the assumptions and keeping them consistent from
start to finish. It is human work, not a technical step, and it is what later lets us read the segments with that same
connotation.

### What you will learn today

1. What business problem a segmentation solves and what a bank does with it.
2. How k-means works, step by step, and why the only thing it sees is distances.
3. The preprocessing that decides the result: binaries to 0/1, what to do with "Desconocido" (unknown), categoricals to scores on declared business axes (and why blind one-hot encoding erases similarity), and scaling.
4. How many groups to ask for: inertia (the elbow) and the silhouette, and what they really measure.
5. How a segment is interpreted: centroids on the axes we defined and in original units, composition, parallel coordinates, and the statistical tests that confirm the groups differ.
6. What happens when you encode blindly with one-hot, and how much the segments depend on the assumptions of the mapping.

The steps in this notebook can also be followed on the course's interactive site (tab "Segmentación de clientes"),
and the **Clustering Playground** in the hub lets you move the centroids by hand to see how k-means converges.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
SEMILLA = 7



## 1. The problem: 5,660 customers and no label

A retail bank has a customer base with what it knows about each one: how old they are, what they do for a living,
their marital status, their education level, and whether they have a savings account, a mortgage or a consumer loan.
With that base the bank wants to do three things, and all three need **groups**:

- **Offers**: there is no point offering a mortgage to someone who already has one, or a college savings plan to a
  retiree. With segments you design a few campaigns, one per group, instead of one per customer.
- **Channel and language**: a 22-year-old student and a 55-year-old manager are not reached through the same channel
  or with the same tone.
- **Risk and provisions**: segments are used to look at delinquency, profitability and churn by customer type, before
  building a predictive model.

What the bank does not have is a column that says "this customer is type 3". That column is what we are going to build.

The columns come with names in Spanish; we rename them so they are easy to type (the names and the data values stay in
Spanish: `edad` = age, `ocupacion` = occupation, `estado_civil` = marital status, `educacion` = education,
`cuenta_ahorro` = savings account, `hipotecario` = mortgage, `credito_consumo` = consumer loan, `impago` = default;
"Si" = yes, "No" = no, "Desconocido" = unknown). If you have the course file (`base_clientes_banco.xlsx`), paste its
link into `URL_DATOS`; if it is left empty, the course copy is downloaded.


In [ ]:

URL_DATOS = ""    # <-- paste the link to the course file (base_clientes_banco.xlsx) here if you have it
URL_RESPALDO = "https://www.dropbox.com/s/lfjqcx9uoey6bap/base_clientes_banco.xlsx?dl=1"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

clientes = pd.read_excel(normalizar_link(URL_DATOS or URL_RESPALDO))
clientes.columns = ["edad", "ocupacion", "estado_civil", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo", "impago"]
print("Customers:", len(clientes), "  Columns:", clientes.shape[1])
clientes.head(8)



Before modeling you have to look at what is in each column. Only `edad` is a number; the other seven are text, and
in several of them the category **"Desconocido"** (unknown) appears, which is not one more category but a missing value
in disguise.


In [ ]:

print(clientes["edad"].describe().round(1).to_string())
print()
for col in clientes.columns[1:]:
    print(f"{col}: " + ", ".join(f"{k} ({v})" for k, v in clientes[col].value_counts().items()))



Three things this table already decides:

- **`impago`** (default) has a single "Si" among 5,660 customers and 1,035 unknowns. A column where everyone has the
  same value separates nobody, and it is also exactly the kind of column you would want to **predict** with the
  segments, not use to build them. We drop it.
- **`hipotecario`** and **`credito_consumo`** have 115 unknowns each (2%). That is few; we will treat them as missing.
- **`educacion`** has a natural order (no education, primary, secondary, vocational course, university). **`ocupacion`**
  and **`estado_civil`** have no single order, but they do have **similarities** that anyone at the bank recognizes: a
  blue-collar worker and a service employee are more alike than a blue-collar worker and a manager. Making those
  similarities explicit is the heart of section 3.



## 2. What k-means does

K-means (MacQueen, 1967; the iterative algorithm is Lloyd's, 1957) looks for $K$ points, the **centroids**, such that
each customer is assigned to the nearest centroid and the sum of squared distances from each customer to its centroid,
the **inertia** (in the literature, the *within-cluster sum of squares*), is as small as possible:

$$\text{inercia} = \sum_{k=1}^{K} \sum_{i \in C_k} \lVert x_i - \mu_k \rVert^2$$

Each symbol: $K$ is the number of groups, which we fix; $C_k$ is the set of customers assigned to group $k$; $x_i$ is
the feature vector of customer $i$ (one row of the numeric matrix we will build in section 3, already scaled); $\mu_k$
is the centroid of group $k$, the average of its customers' vectors; and $\lVert \cdot \rVert^2$ is the squared
Euclidean distance, the sum of the squared differences column by column. In words: the inertia is how far the
customers are, in total, from the center of their own group. In the code the inertia is `modelo.inertia_`, the
centroids `modelo.cluster_centers_` and the assignment `modelo.labels_`. The algorithm is surprisingly simple:

1. Choose $K$ initial centroids (at random, or with the `k-means++` trick that spreads them far apart).
2. **Assign** each customer to the nearest centroid.
3. **Move** each centroid to the average of the customers it received.
4. Repeat 2 and 3 until nobody changes group.

Each round lowers the inertia or leaves it unchanged, so it converges; but it converges to a **local** minimum that
depends on the starting point, which is why scikit-learn runs it several times (`n_init`) and keeps the best one.

The thing to remember is the phrase in step 2: **the nearest centroid**. K-means does not know what an occupation or
an age is; it only computes Euclidean distances between rows of numbers. So everything we decide about how to turn
"Obrero" (blue-collar worker) or "Casado" (married) into numbers, and about the scale of each column, translates
directly into which customers end up together. That is why the next section is the most important one in the notebook.



## 3. Preprocessing: turning customers into vectors without lying

### 3.1 The binaries: yes / no to 1 / 0, and "Desconocido" as missing

A column with two values converts to 0 and 1 without losing anything. The "Desconocido" in `hipotecario` and
`credito_consumo` is marked as missing and imputed with the most frequent value (the mode), which is reasonable when
they are 2% of the rows; with many more it would be better to add an indicator column "unknown".


In [ ]:

datos = clientes.drop(columns="impago").copy()
for col in ["cuenta_ahorro", "hipotecario", "credito_consumo"]:
    datos[col] = datos[col].map({"Si": 1, "No": 0})          # "Desconocido" (unknown) becomes NaN
print("Missing values per column after converting the binaries:")
print(datos.isna().sum().to_string())
for col in ["hipotecario", "credito_consumo"]:
    datos[col] = datos[col].fillna(datos[col].mode()[0]).astype(int)
datos["cuenta_ahorro"] = datos["cuenta_ahorro"].astype(int)



### 3.2 The categoricals: translating similarity into numbers, with declared business logic

This is the most important decision in the notebook, and the one most often done wrong.

The automatic recipe for a text variable is **one-hot encoding** (dummy variables): one 0/1 column per category.
"Casado" (married) becomes (1, 0, 0), "Divorciado" (divorced) (0, 1, 0) and "Soltero" (single) (0, 0, 1). It is quick,
it requires no thought, and it has a cost that is invisible until you look at the segments: **all the categories end up
at the same distance from each other**. The Euclidean distance between two different one-hot vectors is always
$\sqrt{2}$, whatever the pair. For k-means, a married customer is exactly as far from a divorced one as from a single
one; a manager as far from a technician as from a student; and if the variable were the district, Providencia would be
as far from Las Condes as from Puerto Montt (two neighboring districts of Santiago and a city 1,000 km to the south).
All the **similarity** information between values of the same column, obvious to anyone at the bank, is lost. And on
top of that the variable takes up as many dimensions as it has categories, which brings other problems we will see in
section 6.

The alternative is to do the work the algorithm cannot do: define **an axis with business meaning** and assign each
category a **score** on that axis. For marital status, a reasonable axis is the degree of family commitment of the
household, from 0 to 100: Soltero 0, Divorciado 50, Casado 100. With that we tell k-means what we know: that a divorced
person is halfway along (they had a household, probably have children and obligations, and today decide alone), instead
of leaving them equidistant from everything. The value 50 does not come from the data; it comes from our reading of the
business, and that is why it has to be **written down and defended**.

For occupation a single axis is not enough, and it pays to realize that before encoding: a retiree and a student both
have low income, but the retiree's is fixed and secure and the student's is almost nonexistent and unstable. If we put
them together on a single "income" axis, the algorithm would see them as equal. So we use **two axes**, each from 0 to
100:

- **Expected income level** of the occupation: Gerente (manager) 100, Emprendedor (entrepreneur) 75, Empleado
  Independiente (self-employed) 60, Técnico (technician) 60, Administrativo (clerical) 50, Servicios (services) 35,
  Obrero (blue-collar) 30, Jubilado (retiree) 30, Dueña de Casa (homemaker) 10, Estudiante (student) 10, Desempleado
  (unemployed) 0.
- **Income stability** (how predictable and formal it is): Jubilado 100 (a pension does not fail), Administrativo 85,
  Gerente 85, Técnico 80, Obrero 60, Servicios 55, Empleado Independiente 40, Emprendedor 35, Dueña de Casa 30,
  Estudiante 15, Desempleado 0.

Twelve categories become **two** numbers per customer, with a clear reading, instead of twelve unrelated columns.
Education, which already has an order, goes on the same kind of scale so that everything is in the same units: Sin
Educación (no education) 0, Básica (primary) 25, Media (secondary) 50, Curso Profesional (vocational course) 75, Grado
Universitario (university degree) 100 (with this we assume that each educational step is worth the same, another
assumption that is declared).

Three rules so that this mapping is an analysis and not a loose opinion:

1. **Declare the assumptions** in a table, before running anything, with the logic of each axis. That is what we do in
   the next cell.
2. **Keep them consistent** throughout the analysis: the same scores for segmenting, for interpreting the centroids and
   for assigning new customers. Changing the scale midway invalidates everything before it.
3. **Read the results with that connotation**: if a segment has "stability 80", that means "high on the stability axis
   that we defined", not a measured fact. And in section 6 we will measure how much the segments change when an
   assumption changes, which is the honest way to find out what the segmentation depends on.

The "Desconocido" values in the three variables (37 in occupation, 13 in marital status, 282 in education) are missing
and are imputed with the median score, which leaves them at the center of the axis, at the same distance from both
extremes.


In [ ]:

# Assumptions table: each category receives a score from 0 to 100 on a declared business axis.
# These scores are our decisions, not data; they are used unchanged throughout the notebook.
PUNTAJE_INGRESO = {"Gerente": 100, "Emprendedor": 75, "Empleado Independiente": 60, "Técnico": 60, "Administrativo": 50,
                   "Servicios": 35, "Obrero": 30, "Jubilado": 30, "Dueña de Casa": 10, "Estudiante": 10, "Desempleado": 0}
PUNTAJE_ESTABILIDAD = {"Jubilado": 100, "Administrativo": 85, "Gerente": 85, "Técnico": 80, "Obrero": 60, "Servicios": 55,
                       "Empleado Independiente": 40, "Emprendedor": 35, "Dueña de Casa": 30, "Estudiante": 15, "Desempleado": 0}
PUNTAJE_COMPROMISO = {"Soltero": 0, "Divorciado": 50, "Casado": 100}
PUNTAJE_EDUCACION = {"Sin Educación": 0, "Básica": 25, "Media": 50, "Curso Profesional": 75, "Grado Universitario": 100}

supuestos = pd.DataFrame({"axis": ["ingreso", "estabilidad", "compromiso", "educacion"],
                          "original_variable": ["ocupacion", "ocupacion", "estado_civil", "educacion"],
                          "what_it_measures": ["expected income level of the occupation", "how predictable and formal that income is",
                                       "degree of family commitment of the household", "years of formal education, equal steps"],
                          "extremes": ["Desempleado 0 ... Gerente 100", "Desempleado 0 ... Jubilado 100", "Soltero 0, Divorciado 50, Casado 100",
                                       "Sin Educación 0 ... Universitario 100"]})
print(supuestos.to_string(index=False))

datos["ingreso"] = datos["ocupacion"].map(PUNTAJE_INGRESO)              # "Desconocido" becomes NaN
datos["estabilidad"] = datos["ocupacion"].map(PUNTAJE_ESTABILIDAD)
datos["compromiso"] = datos["estado_civil"].map(PUNTAJE_COMPROMISO)
datos["educacion"] = datos["educacion"].map(PUNTAJE_EDUCACION)
for col in ["ingreso", "estabilidad", "compromiso", "educacion"]:
    datos[col] = datos[col].fillna(datos[col].median())

COLUMNAS_MODELO = ["edad", "ingreso", "estabilidad", "compromiso", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]
X = datos[COLUMNAS_MODELO].astype(float)
print()
print("Matrix for k-means:", X.shape[0], "customers x", X.shape[1], "columns")
X.head()



Before moving on, it is worth seeing in numbers what changed. The next cell computes the distance between pairs of
customers that differ in a single category, under the two encodings. With one-hot all distances between different
categories are the same; with the mapping, the distance reflects the similarity we declared.


In [ ]:

def distancia_onehot(a, b):
    return 0.0 if a == b else np.sqrt(2)

pares = [("Casado", "Divorciado"), ("Casado", "Soltero"), ("Divorciado", "Soltero")]
print("Marital status: distance between two customers identical except in this column")
for a, b in pares:
    print(f"  {a:10s} vs {b:10s}   one-hot: {distancia_onehot(a, b):.2f}   mapping (commitment axis): {abs(PUNTAJE_COMPROMISO[a] - PUNTAJE_COMPROMISO[b]):>5.0f}")
print()
pares = [("Gerente", "Técnico"), ("Gerente", "Estudiante"), ("Obrero", "Servicios"), ("Jubilado", "Estudiante"), ("Jubilado", "Obrero")]
print("Occupation: the same, with the two axes (Euclidean distance on the income x stability plane)")
for a, b in pares:
    d = np.hypot(PUNTAJE_INGRESO[a] - PUNTAJE_INGRESO[b], PUNTAJE_ESTABILIDAD[a] - PUNTAJE_ESTABILIDAD[b])
    print(f"  {a:10s} vs {b:10s}   one-hot: {distancia_onehot(a, b):.2f}   mapping: {d:>6.1f}")

ocupaciones = pd.DataFrame({"ocupacion": list(PUNTAJE_INGRESO), "ingreso": list(PUNTAJE_INGRESO.values()),
                            "estabilidad": [PUNTAJE_ESTABILIDAD[o] for o in PUNTAJE_INGRESO],
                            "clientes": [int((clientes["ocupacion"] == o).sum()) for o in PUNTAJE_INGRESO]})
fig = px.scatter(ocupaciones, x="ingreso", y="estabilidad", size="clientes", text="ocupacion", size_max=45,
                 title="The map of occupations we hand to k-means (size = number of customers)")
fig.update_traces(textposition="top center")
fig.update_layout(height=480, width=760, xaxis_range=[-10, 115], yaxis_range=[-10, 115])
fig.show()



The plot is the segmentation of occupations that **we** made before any algorithm ran, and it is worth looking at it
with that awareness: k-means will inherit these proximities. Retiree and student, which one-hot put at the same distance
as any other pair, are now at opposite corners of the stability axis; blue-collar and services are almost on top of
each other. If someone at the bank disagrees with a score, this is the moment to argue about it, not after seeing the
segments.



### 3.3 Scaling: no column should dominate because of its unit

Age runs from 17 to 92, the scores from 0 to 100 and the products from 0 to 1. Without doing anything, a 50-point
difference in income would weigh 2,500 times more in the distance (squared) than having or not having a savings
account, by pure accident of units. We standardize each column (subtract its mean and divide by its standard
deviation) so that they all weigh about the same: $z_{ij} = (x_{ij} - \bar{x}_j) / s_j$, where $x_{ij}$ is the value of
customer $i$ in column $j$, $\bar{x}_j$ the column average and $s_j$ its sample standard deviation. A $z$ of +2 reads
"two standard deviations above the average customer". Notice that scaling does not change the proximities **within** an
axis: divorced is still halfway between single and married; it only changes how much that axis weighs against the
others. It is the same `StandardScaler` from the regression notebooks, with one difference: here there is no training
and testing to separate, because we are not going to predict anything about new customers (if later on new customers
had to be assigned to the segments, the already fitted scaler would be used, just as before).


In [ ]:

escalador = StandardScaler().fit(X)
Z = pd.DataFrame(escalador.transform(X), columns=X.columns, index=X.index)
print("After scaling, each column has mean 0 and standard deviation 1:")
print(Z.describe().loc[["mean", "std"]].round(2).iloc[:, :8].to_string())



## 4. How many groups: the elbow and the silhouette

K-means needs us to tell it $K$, and there is no correct answer hidden in the data. There are two criteria that help,
and it is worth knowing what each one measures.

**Inertia** always goes down when $K$ goes up (with $K$ equal to the number of customers it is zero), so you cannot
pick the minimum. What you look for is the **elbow**: the point beyond which adding one more group barely reduces it.

**The silhouette** (Rousseeuw, 1987) of a customer compares how close it is to its own group against how close it is to
the nearest neighboring group:

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$

where $s_i$ is the silhouette of customer $i$, $a_i$ is the average distance from customer $i$ to the other customers in
its own group (how compact its group is as seen from that customer) and $b_i$ the average distance to the customers of
the nearest group that is not its own (how far away the alternative is). The maximum in the denominator keeps the ratio
between -1 and 1. In words: how much closer the customer is to its group than to the neighboring group, in proportion.
It is close to 1 if the customer is clearly in the right group, close to 0 if it is on the boundary, and negative if it
is probably in the wrong group. The average over all customers is the silhouette of the clustering, and here you do
look for the **maximum**. As a reference, above 0.5 people talk about reasonable structure and below 0.25 about weak
structure; with real customer data it rarely goes above 0.3, and that is not a failure: it means the customers form a
continuum, not islands.


In [ ]:

ks = range(2, 11)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Z, km.labels_, sample_size=3000, random_state=SEMILLA))
criterios = pd.DataFrame({"K": list(ks), "inercia": inercias, "silueta": siluetas}).set_index("K")
print(criterios.round(3).to_string())

fig = go.Figure()
fig.add_scatter(x=criterios.index, y=criterios["inercia"], mode="lines+markers", name="inertia")
fig.add_scatter(x=criterios.index, y=criterios["silueta"], mode="lines+markers", name="silhouette", yaxis="y2")
fig.update_layout(title="How many groups: inertia (look for the elbow) and silhouette (look for the maximum)", xaxis_title="K",
                  yaxis=dict(title="inertia"), yaxis2=dict(title="silhouette", overlaying="y", side="right"), height=420, width=760)
fig.show()



Inertia goes down evenly and shows no clear elbow; the silhouette rises quickly up to $K = 5$ (from 0.17 to 0.20) and
then oscillates between 0.19 and 0.21 up to $K = 10$, with no clear maximum. This is the typical situation: the
criteria guide, they do not decide. In practice the number of segments is also chosen by what the business can handle:
five different campaigns are workable, fifteen are not. We will work with **K = 5**.


In [ ]:

K = 5
modelo = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z)
datos["segmento"] = modelo.labels_
clientes["segmento"] = modelo.labels_
print("Inertia:", round(modelo.inertia_), "  Silhouette:", round(silhouette_score(Z, modelo.labels_, sample_size=3000, random_state=SEMILLA), 3))
print()
print("Customers per segment:")
print(datos["segmento"].value_counts().sort_index().to_string())



## 5. Interpreting the segments

The model delivered one label per customer and nothing else. Interpreting means going back to units we understand: for
each segment, the average age, the average score on each axis we defined (income, stability, commitment, education),
the percentage holding each product, and the composition by original occupation and marital status. The centroids on
the standardized scale are useful to see which column "pulls" each group (a value of +2 means two standard deviations
above the overall average), but the segment's name comes from the real units.

And here the work of section 3 pays off: since we defined the axes, the segments are read **in those terms**. "Income
63, stability 79" is not a measured fact; it means "occupations that we placed high on income and on stability". The
interpretation inherits the assumptions, and that is what makes it defensible in front of management: every number in
the table can be traced back to a declared decision.


In [ ]:

perfil = datos.groupby("segmento").agg(clientes=("edad", "size"), edad=("edad", "mean"), ingreso=("ingreso", "mean"), estabilidad=("estabilidad", "mean"),
                                       compromiso=("compromiso", "mean"), educacion=("educacion", "mean"),
                                       pct_ahorro=("cuenta_ahorro", "mean"), pct_hipotecario=("hipotecario", "mean"), pct_consumo=("credito_consumo", "mean"))
perfil["pct_del_total"] = perfil["clientes"] / len(datos)
print("Profile of each segment (scores from 0 to 100 on the defined axes; products as a fraction):")
print(perfil.round(2).to_string())
print()
for col in ["ocupacion", "estado_civil", "educacion"]:
    print(f"Composition by {col} (row = segment, fraction of its customers):")
    print(pd.crosstab(clientes["segmento"], clientes[col], normalize="index").round(2).to_string())
    print()



Reading of the five segments (the segment numbers are arbitrary and may change order on a rerun; what matters is the
profile):

- **No income of their own** (402 customers, 7%): 33 years old, income 6 and stability 11 on our axes, that is,
  students (46%), unemployed (41%) and homemakers (12%), mostly single, and with the **highest savings account
  ownership** (56%). A group that one-hot would never have formed, because it joins three different occupations that
  share an economic situation.
- **Married professionals** (1,415, 25%): 42 years old, income 63, stability 79, commitment 92, the highest education
  (87): clerical staff, technicians and managers with an established household. The segment for mortgages and
  investment products.
- **Low-income married** (1,797, 32%): 45 years old, income 37 and the lowest education (36): blue-collar workers
  (44%), services, clerical staff with primary education and retirees. The largest, and the one with the lowest savings
  account ownership (22%).
- **With a consumer loan** (810, 14%): the only segment with a consumer loan (100% against 0% in the others), and
  otherwise a mix of everything. A single binary column defined it, and in section 6 we explain why.
- **Working singles** (1,236, 22%): 33 years old, commitment 2 (96% single), medium income and stability, education 70:
  the young base still in formation, with 38% holding a savings account.

Compare this with what would have come out without the mapping (we will run it in section 6): groups that are each a
single occupation. Here the segments cut across occupations because we told the algorithm which occupations are alike.


In [ ]:

# Centroids on the standardized scale: which columns define each segment
centroides = pd.DataFrame(modelo.cluster_centers_, columns=Z.columns)
centroides.insert(0, "segmento", range(K))
fig = px.parallel_coordinates(centroides, color="segmento", dimensions=list(Z.columns),
                              title="Centroids in standardized units (0 = overall average; +2 = two standard deviations above)")
fig.update_layout(height=480)
fig.show()


In [ ]:

fig = px.scatter(datos.assign(segmento=datos["segmento"].astype(str), ocupacion=clientes["ocupacion"], estado_civil=clientes["estado_civil"],
                              ingreso_j=datos["ingreso"] + np.random.default_rng(0).uniform(-3, 3, len(datos)),
                              estabilidad_j=datos["estabilidad"] + np.random.default_rng(1).uniform(-3, 3, len(datos))),
                 x="ingreso_j", y="estabilidad_j", color="segmento", opacity=0.45, hover_data=["ocupacion", "estado_civil", "edad"],
                 labels={"ingreso_j": "income (defined axis, with jitter to see the points)", "estabilidad_j": "stability (defined axis)"},
                 title="The customers on the occupation plane we defined, colored by segment")
fig.update_layout(height=500, width=820)
fig.show()
comp = pd.crosstab(clientes["segmento"], clientes["ocupacion"], normalize="index").reset_index().melt(id_vars="segmento", var_name="ocupacion", value_name="fraccion")
fig = px.bar(comp, x="segmento", y="fraccion", color="ocupacion", title="Composition of each segment by occupation", barmode="stack")
fig.update_layout(height=420, width=820)
fig.show()



### The tests: do the segments really differ?

The plot suggests differences; statistics confirms them. For a numeric variable like age or a score, one-way **ANOVA**
(Fisher, 1925) compares the means across the $K$ segments with the $F$ statistic, the ratio between the variance of the
group means and the variance within the groups: the null hypothesis is that all the means are equal, and a small
p-value rejects it. For a categorical variable like the original occupation, the test is the **chi-squared test of
independence** (Pearson, 1900) on the segment by category contingency table: it compares the observed frequencies with
those there would be if the distribution of occupations were the same in every segment.

An implementation detail that often goes wrong: `f_oneway` takes **one sample per group** (the ages of the customers
in segment 0, those in segment 1, and so on), not the full column together with the label.

And an honest warning: with 5,660 customers almost any difference comes out significant, and besides, the groups were
built precisely so that they differ on these columns, so a small p-value here is almost inevitable. The test is useful
for spotting the column that does **not** separate (the one that can be dropped from the analysis), more than for
celebrating the ones that do.


In [ ]:

filas = []
for col in COLUMNAS_MODELO:
    muestras = [datos.loc[datos["segmento"] == k, col] for k in range(K)]
    F, p = stats.f_oneway(*muestras)
    filas.append(dict(variable=col, prueba="ANOVA", estadistico=F, valor_p=p))
for col in ["ocupacion", "estado_civil", "educacion"]:
    tabla = pd.crosstab(clientes["segmento"], clientes[col])
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    filas.append(dict(variable=col + " (original)", prueba="chi-squared", estadistico=chi2, valor_p=p))
pruebas = pd.DataFrame(filas)
pruebas["valor_p"] = pruebas["valor_p"].map(lambda v: f"{v:.2e}")
print(pruebas.round(1).to_string(index=False))



The only column that does not separate is `hipotecario` (p-value of 0.42): mortgage ownership is practically the same
in the five segments (56% to 60%). For a mortgage campaign these segments are useless as they stand, and that is useful
information: you would have to segment with other columns or directly build a propensity model.



## 6. What happens if you encode blindly, and what the segments depend on

### 6.1 One-hot: the segments become categories

We repeat exactly the same procedure (binaries, education on a scale, standardize, k-means with K = 5) but with
`ocupacion` and `estado_civil` one-hot encoded, which is what anyone does who does not stop to think about similarity.
The matrix goes from 8 columns to 19.


In [ ]:

X_onehot = pd.get_dummies(datos[["edad", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]].join(clientes[["ocupacion", "estado_civil"]]),
                          columns=["ocupacion", "estado_civil"], dtype=float)
X_onehot = X_onehot.drop(columns=[c for c in X_onehot.columns if c.endswith("_Desconocido")])
Z_onehot = StandardScaler().fit_transform(X_onehot)
modelo_onehot = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z_onehot)
clientes["segmento_onehot"] = modelo_onehot.labels_
from sklearn.metrics import adjusted_rand_score
print("Columns:", X_onehot.shape[1], "  Silhouette:", round(silhouette_score(Z_onehot, modelo_onehot.labels_, sample_size=3000, random_state=SEMILLA), 3),
      "  Adjusted Rand index against the mapped segmentation:", round(adjusted_rand_score(clientes["segmento"], clientes["segmento_onehot"]), 3))
print()
print("Composition by occupation of the one-hot segments:")
print(pd.crosstab(clientes["segmento_onehot"], clientes["ocupacion"], normalize="index").round(2).to_string())
print()
print("Composition by marital status:")
print(pd.crosstab(clientes["segmento_onehot"], clientes["estado_civil"], normalize="index").round(2).to_string())



Three of the five segments are **a single category**: "the students" (100%), "the retirees" (100%), "the blue-collar
workers" (91%). The other two are the rest of the base split by marital status. The silhouette is similar (0.18
against 0.20), so the numeric criterion gives no warning; the problem is visible only when you look at the composition.
The adjusted Rand index (Hubert and Arabie, 1985), which is 1 if two partitions coincide and 0 if they coincide as much
as two random partitions would, gives 0.31: they are different segmentations of the same base.

What happened has two parts. The first is the one we already discussed: one-hot erased the similarity, so the
algorithm had no way to put a student together with an unemployed person or a blue-collar worker with a service
employee. The second is mechanical and worth understanding: a rare category, say "Estudiante" with 186 customers out
of 5,660, becomes a column with 3% ones. When it is standardized, that 1 turns into a value close to +5.4 (because the
standard deviation of such an unbalanced column is very small), while the 0 stays close to -0.2. The 186 customers
with that 1 end up **extremely far** from everyone else along that dimension, and the cheapest thing in inertia is to
give them their own centroid. The more categories a nominal has, the more dimensions it contributes, the more it
dominates, and the more the result looks like a frequency table than a segmentation. It is the **curse of
dimensionality** applied to clustering: with many 0/1 columns every point is far from every other and distance stops
discriminating.

The same mechanism explains the "with a consumer loan" segment of our segmentation: `credito_consumo` is a binary with
14% ones, and once standardized it strongly separates those who have it. There the decision is a business one: if the
bank wants a product to define a segment (it is actionable), leave it in; if not, take it out of the segmenting
columns and use it to describe, as we do in exercise 2.

### 6.2 Sensitivity: how much the segments depend on our assumptions

If the scores are decisions, the mandatory question is how much the result would change with other reasonable
decisions. We try three changes, one at a time, and measure the adjusted Rand index against the base segmentation:

1. Put divorced at 0 instead of 50 (read "divorced" as "today decides alone", no commitment).
2. Raise the retiree's income from 30 to 60 (read retirement as a stage with accumulated wealth).
3. Remove the stability axis and leave occupation with a single income axis.


In [ ]:

def segmentar(tabla, columnas, k=K):
    Zs = StandardScaler().fit_transform(tabla[columnas].astype(float))
    return KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(Zs).labels_

variantes = {}
d1 = datos.copy(); d1["compromiso"] = clientes["estado_civil"].map({"Soltero": 0, "Divorciado": 0, "Casado": 100}).fillna(50)
variantes["divorced = 0"] = segmentar(d1, COLUMNAS_MODELO)
d2 = datos.copy(); d2["ingreso"] = clientes["ocupacion"].map({**PUNTAJE_INGRESO, "Jubilado": 60}).fillna(datos["ingreso"].median())
variantes["retiree income 60"] = segmentar(d2, COLUMNAS_MODELO)
variantes["no stability axis"] = segmentar(datos, [c for c in COLUMNAS_MODELO if c != "estabilidad"])

sensibilidad = pd.DataFrame({"variant": list(variantes), "adjusted_rand_vs_base": [adjusted_rand_score(datos["segmento"], v) for v in variantes.values()]})
print(sensibilidad.round(3).to_string(index=False))
print()
print("What happens to the retirees when their income rises to 60 (row = base segment, column = new segment):")
jub = clientes["ocupacion"] == "Jubilado"
print(pd.crosstab(datos.loc[jub, "segmento"], variantes["retiree income 60"][jub.values]))



All three changes move the segmentation (adjusted Rand index between 0.69 and 0.77: most customers keep the same
companions, a minority changes group), and each one moves it where you would expect: moving the divorced changes the
group of the divorced, raising the retiree's income regroups 239 of the 295 retirees into one new segment (the table
below shows it), and removing the stability axis mixes students with retirees again. That is exactly what we want from
a segmentation with declared assumptions: being able to say "if the bank prefers to read retirement this other way,
these 295 customers change and nothing else". With one-hot there are no assumptions to argue about, but there is no
similarity either: the decision does not disappear, it hides.

## 7. What to take away

1. Clustering is learning without a label: the model proposes groups and we do the interpretation, in units we understand.
2. K-means only sees Euclidean distances. Preprocessing is not a preliminary formality: it is where the result is decided.
3. Blindly one-hot encoding a categorical erases all similarity between its values (married and divorced end up as far apart as married and single) and multiplies the dimensions. Instead, define axes with business meaning and translate each category into a score: it is human work, with assumptions that are declared beforehand, kept consistent throughout the analysis and used to read the results.
4. A variable may need more than one axis (occupation: income and stability). Think about the axes before encoding.
5. Binaries to 0/1; "Desconocido" is missing, not a category; scale at the end.
6. Inertia looks for the elbow and the silhouette for the maximum; both guide and the business decides. Silhouettes of 0.2 to 0.3 are normal with real customers, and a similar silhouette does not mean a similar segmentation.
7. Measure sensitivity to the assumptions with the adjusted Rand index: a good segmentation changes where the assumption was changed, and nowhere else.
8. ANOVA and chi-squared confirm differences, but with thousands of customers almost everything is significant: use them to find what does not separate.



## Exercises

1. **Your own mapping.** Propose another score table for `ocupacion` (for example, a "life stage" axis instead of
   stability, or a different income ordering), justify each value in one sentence, segment again with K = 5 and compare
   with the base segmentation using the adjusted Rand index and the composition by occupation. Which customers changed
   group? Is it what you expected given your change?
2. **Segment with some columns and describe with others.** Take `credito_consumo` and `hipotecario` out of the
   segmenting columns (keep them to describe), rerun k-means with K = 5 and check whether the segment defined by the
   consumer loan disappears. Then compute the percentage with a consumer loan in each new segment: are there segments
   where the product is naturally concentrated?
3. **Stability.** A segmentation is useful if it does not change when the data change a little. Split the base at
   random into two halves, run k-means with K = 5 on each, assign the customers of the second half to the centroids of
   the first, and measure the adjusted Rand index between that assignment and the second half's own segmentation.



## Solutions


In [ ]:

# Exercise 1: another mapping for occupation ("life stage" axis instead of stability)
# Assumption: life stage orders the occupations by the typical moment in a career, from 0 (training) to 100 (retirement).
PUNTAJE_ETAPA = {"Estudiante": 0, "Desempleado": 30, "Servicios": 35, "Obrero": 45, "Administrativo": 45, "Técnico": 50, "Empleado Independiente": 55,
                 "Dueña de Casa": 55, "Emprendedor": 60, "Gerente": 70, "Jubilado": 100}
d_ej = datos.copy()
d_ej["etapa"] = clientes["ocupacion"].map(PUNTAJE_ETAPA).fillna(50)
cols_ej = ["edad", "ingreso", "etapa", "compromiso", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]
seg_ej = segmentar(d_ej, cols_ej)
print("Adjusted Rand index against the base segmentation:", round(adjusted_rand_score(datos["segmento"], seg_ej), 3))
print()
print("Composition by occupation with the life stage axis:")
print(pd.crosstab(seg_ej, clientes["ocupacion"], normalize="index").round(2).to_string())
print()
print("Who changed group (row = base segment, column = new):")
print(pd.crosstab(datos["segmento"], seg_ej))



Changing the second occupation axis changes the segmentation moderately (adjusted Rand index 0.71) and, once again,
where it should: with the life stage axis the retirees, who sit alone at the 100 end, form a segment of their own (98%
retirees), and the students, who on the stability axis shared a group with the unemployed and homemakers, are spread
across the young segments. The life stage axis is also highly correlated with age, so it adds less new information than
the stability axis. The lesson is not which mapping is "the right one", but that the chosen mapping must be explainable
and that its consequences can be measured.


In [ ]:

# Exercise 2: segment without the credit products and use them to describe
cols_sin_credito = ["edad", "ingreso", "estabilidad", "compromiso", "educacion", "cuenta_ahorro"]
seg_sin = segmentar(datos, cols_sin_credito)
clientes["segmento_sin_credito"] = seg_sin
print("Adjusted Rand index against the base:", round(adjusted_rand_score(datos["segmento"], seg_sin), 3))
print()
resumen = datos.assign(segmento_sin_credito=seg_sin).groupby("segmento_sin_credito").agg(
    clientes=("edad", "size"), edad=("edad", "mean"), ingreso=("ingreso", "mean"), estabilidad=("estabilidad", "mean"), compromiso=("compromiso", "mean"),
    pct_consumo=("credito_consumo", "mean"), pct_hipotecario=("hipotecario", "mean"))
print(resumen.round(2).to_string())



Without the credit products in the segmentation, the "with a consumer loan" segment disappears and the five groups are
defined by the person's profile; the consumer loan spreads across them with little variation, which confirms that in
this base the product does not follow the demographic profile. For the bank it is the difference between "who are my
customers?" (segmenting without products) and "to whom do I offer what?" (segmenting with them); both questions are
legitimate and it pays to know which one you are answering.


In [ ]:

# Exercise 3: stability between two halves
rng = np.random.default_rng(SEMILLA)
idx = rng.permutation(len(Z))
a, b = idx[: len(Z) // 2], idx[len(Z) // 2:]
ma = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z.iloc[a])
mb = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z.iloc[b])
asignacion_desde_a = ma.predict(Z.iloc[b])          # the customers of half b, assigned to the centroids of a
print("Adjusted Rand index between half b's own segmentation and the one inherited from half a:",
      round(adjusted_rand_score(mb.labels_, asignacion_desde_a), 3))



A high adjusted Rand index between the two halves means the segments do not depend on which customers landed in the
sample: they are a feature of the base, not of chance. This is the stability test that really matters, because a
segmentation that changes with every draw is useless for designing campaigns that last months. Together with the
sensitivity to the assumptions in section 6.2, it forms the pair of questions to answer before delivering a
segmentation: does it depend on the sample? and does it depend on my decisions, and on which ones?
